#Chat Metrics

##Data Collection

In [1]:
import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import graphviz

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  classification_report
)
from sklearn import tree

df = pd.read_parquet('https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet')

##XGBoost

In [ ]:
features = [
  'output_tokens',
  'month_sin',
  'model_calls',
  'primary_category_navigation',
  'answer_sentiment',
  'question_length',
  'hour_sin',
  'total_tokens',
  'primary_category_sunport_amenities',
  'day_sin'
]

X = df[features]
y = df['satisfaction']

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20,
  random_state=42,
  stratify=y #Maintains roughly same proportion
)

#Final XGBoost model
xgb = XGBClassifier(
  n_estimators=300,   #Boosting rounds/trees
  learning_rate=0.01, #Controls how much each new tree contributest to model
  max_depth=8,        #Max deptho or each tree
  random_state=42,
  objective = 'binary:logistic', # Tells XGBoost to do binary classification
  eval_metric='logloss'
)

#Fit
xgb.fit(X_train, y_train)

#Predict
y_pred_xgb = xgb.predict(X_test)

#Probability
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_xgb))
print('Precision:', precision_score(y_test, y_pred_xgb))
print('Recall:', recall_score(y_test, y_pred_xgb))
print('F1:', f1_score(y_test, y_pred_xgb))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_xgb))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb))

##Data Visualization

In [ ]:
#Create df for XGBoost importance features
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': xgb.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))

#Create horizontal barplot
sns.barplot(
    data=feature_importance.head(10),
    x='importance',
    y='feature',
    hue='feature',
    palette='viridis',
    legend=False
)

plt.title('Top 10 Feature Importances - XGBoost', fontsize=14)
plt.xlabel('Relative Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)

plt.tight_layout()

plt.savefig('important_features.png', dpi=300, bbox_inches='tight')

plt.show()

##Communication of Results